In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 8.4 Systematic Model Comparison Framework
- Extends 4.1: Logistic, Decision Tree, Random Forest, XGBoost (administrative) + Survey-Enhanced XGBoost
- Full metric suite + equity/subgroup breakdown

## Setup

In [ ]:
import numpy as np
import pandas as pd
import time
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss, log_loss, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train_df = pd.read_csv('../data/ML_SURVEY_MASTER_TRAIN.csv')
test_df = pd.read_csv('../data/ML_SURVEY_MASTER_TEST.csv')
print("Training shape:", train_df.shape, "| Testing shape:", test_df.shape)

## Administrative Feature Set (matches original 4.1 models)

In [ ]:
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA', 'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1', 'UNITS_COMPLETED_2', 'DFW_UNITS_1', 'DFW_UNITS_2', 'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2', 'GRADE_POINTS_1', 'GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY', 'GENDER', 'FIRST_GEN_STATUS', 'COLLEGE']
numeric_features = [c for c in numeric_features if c in train_df.columns]
categorical_features = [c for c in categorical_features if c in train_df.columns]

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_admin_medians = train_enc.median()
train_enc = train_enc.fillna(train_admin_medians)
test_enc = test_enc.fillna(train_admin_medians)

X_train_admin, y_train = train_enc, train_df['DEPARTED']
X_test_admin, y_test = test_enc, test_df['DEPARTED']

scaler = StandardScaler()
X_train_admin_scaled = scaler.fit_transform(X_train_admin)
X_test_admin_scaled = scaler.transform(X_test_admin)
print("Administrative features:", X_train_admin.shape[1])

## Survey-Enhanced Feature Set (the full master matrix)

In [ ]:
X_train_survey = train_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])
X_test_survey = test_df.drop(columns=['DEPARTED', 'SEM_3_STATUS'])
X_train_survey.columns = X_train_survey.columns.astype(str)
X_test_survey.columns = X_test_survey.columns.astype(str)
X_train_survey, X_test_survey = X_train_survey.align(X_test_survey, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_survey_medians = X_train_survey.median(numeric_only=True)
X_train_survey = X_train_survey.fillna(train_survey_medians)
X_test_survey = X_test_survey.fillna(train_survey_medians)
print("Survey-enhanced features:", X_train_survey.shape[1])

## Load Tuned Models (hyperparameters from Module 3, refit here)

In [ ]:
dt = pickle.load(open('../models/dt_tuned_f1.pkl', 'rb'))
rf = pickle.load(open('../models/rf_tuned_f1.pkl', 'rb'))
xgb_admin = pickle.load(open('../models/xgb_tuned_f1.pkl', 'rb'))
xgb_survey = pickle.load(open('../models/xgb_tuned_f1.pkl', 'rb'))
lr = LogisticRegression(penalty='l1', solver='saga', C=0.01, max_iter=1000, random_state=RANDOM_STATE)

## Train All Five Models

In [ ]:
trained_models = {}

for name, model, X_tr, X_te, feature_set in [
    ('Regularized Logistic', lr, X_train_admin, X_test_admin, 'Administrative'),
    ('Decision Tree', dt, X_train_admin, X_test_admin, 'Administrative'),
    ('Random Forest', rf, X_train_admin, X_test_admin, 'Administrative'),
    ('XGBoost', xgb_admin, X_train_admin, X_test_admin, 'Administrative'),
    ('Survey-Enhanced XGBoost', xgb_survey, X_train_survey, X_test_survey, 'Survey-enhanced'),
]:
    start = time.time()
    model.fit(X_tr, y_train)
    trained_models[name] = {'model': model, 'X_test': X_te, 'y_test': y_test,
                             'train_time': time.time() - start, 'feature_set': feature_set}

print("All five models trained.")

## Evaluate All Models

In [ ]:
def evaluate_binary_classifier(name, info):
    model, X_test, y_test = info['model'], info['X_test'], info['y_test']
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name, 'Feature Set': info['feature_set'],
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1 Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob),
        'Avg Precision': average_precision_score(y_test, prob),
        'Brier Score': brier_score_loss(y_test, prob),
        'Log Loss': log_loss(y_test, prob),
        'Train Time (s)': info['train_time']
    }

results_df = pd.DataFrame([evaluate_binary_classifier(n, i) for n, i in trained_models.items()]).set_index('Model')
print(results_df.round(4).to_string())

## Feature-Group Importance for the Survey-Enhanced Model

In [ ]:
academic_prefixes = ['HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA', 'GPA_', 'DFW_RATE_', 'DFW_UNITS_',
                     'UNITS_ATTEMPTED_', 'UNITS_COMPLETED_', 'GRADE_POINTS_']
demo_prefixes = ['GENDER_', 'RACE_ETHNICITY_', 'FIRST_GEN_STATUS_', 'COLLEGE_']
text_prefixes = ['TEXT_PC']

def assign_feature_group(col):
    col = str(col)
    if any(col.startswith(p) for p in academic_prefixes):
        return 'Academic performance'
    if any(col.startswith(p) for p in demo_prefixes):
        return 'Demographics'
    if any(col.startswith(p) for p in text_prefixes):
        return 'Survey/Text components'
    return 'Other / check'

xgb_survey_fi = pd.DataFrame({'Feature': X_train_survey.columns, 'Importance': xgb_survey.feature_importances_})
xgb_survey_fi['Feature Group'] = xgb_survey_fi['Feature'].apply(assign_feature_group)
group_importance = xgb_survey_fi.groupby('Feature Group', as_index=False)['Importance'].sum().sort_values('Importance', ascending=False)
print(group_importance)

## Summary
- Same core five-model comparison as the lecture: Logistic, Decision Tree, Random Forest, XGBoost (administrative) + Survey-Enhanced XGBoost.
- Both feature sets use train-only medians for imputation — no leakage.
- Group importance answers whether survey/text data actually adds predictive signal beyond academics and demographics.

**Next:** 8.5 deploys the recommended model to new students.